In [1]:
# ==============================================================
# CELL 1: THÊM CÁC THƯ VIỆN CẦN THIẾT
# ==============================================================
import os
import numpy as np
import pandas as pd

# Đường dẫn thư mục làm việc gốc
BASE_DIR = os.getcwd()

RAW_FILENAME = "flights.csv"
AIRPORTS_FILENAME = "airports.csv"
BTS_FILENAME = "T_MASTER_CORD.csv"
CLEANED_FILENAME = "flights_cleaned.csv"

AIRLINE_CODE = "DL"  # Hãng hàng không cần lọc (ví dụ: DL - Delta Air Lines)
OUTPUT_AIRLINE_FILENAME = f"flights_{AIRLINE_CODE}.csv"

RAW_PATH = os.path.join(BASE_DIR, RAW_FILENAME)
AIRPORTS_PATH = os.path.join(BASE_DIR, AIRPORTS_FILENAME)
BTS_PATH = os.path.join(BASE_DIR, BTS_FILENAME)
CLEANED_PATH = os.path.join(BASE_DIR, CLEANED_FILENAME)
OUTPUT_AIRLINE_PATH = os.path.join(BASE_DIR, OUTPUT_AIRLINE_FILENAME)

In [2]:
# ==============================================================
# CELL 2: ĐỌC DỮ LIỆU THÔ VÀ DỮ LIỆU DANH MỤC SÂN BAY
# ==============================================================
def load_flight_data(path: str) -> pd.DataFrame:
    """Đọc file dữ liệu chuyến bay, tự động xử lý định dạng."""
    try:
        return pd.read_csv(
            path, dtype={"ORIGIN_AIRPORT": str, "DESTINATION_AIRPORT": str}
        )
    except UnicodeDecodeError:
        return pd.read_excel(path, engine="xlrd")


# Đọc danh sách IATA hợp lệ
airports = pd.read_csv(AIRPORTS_PATH)
VALID_AIRPORT_CODES = set(airports["IATA_CODE"])

# Đọc dữ liệu flights gốc
df = load_flight_data(RAW_PATH)
print(f"[1] Đã đọc {df.shape[0]:,} dòng, {df.shape[1]} cột từ '{RAW_FILENAME}'.")

[1] Đã đọc 5,819,079 dòng, 31 cột từ 'flights.csv'.


In [3]:
# ==============================================================
# CELL 3: TẠO CỘT NGÀY ĐẦY ĐỦ (FLIGHT_DATE)
# ==============================================================
df["FLIGHT_DATE"] = pd.to_datetime(
    dict(year=df["YEAR"], month=df["MONTH"], day=df["DAY"])
)
print(
    f"[2] Đã tạo cột FLIGHT_DATE từ YEAR/MONTH/DAY. Ví dụ ngày đầu tiên: {df['FLIGHT_DATE'].iloc[0].date()}"
)

[2] Đã tạo cột FLIGHT_DATE từ YEAR/MONTH/DAY. Ví dụ ngày đầu tiên: 2015-01-01


In [4]:
# ==============================================================
# CELL 4: GIẢI MÃ VÀ CHUẨN HÓA MÃ SÂN BAY (MÃ SỐ DOT -> MÃ IATA)
# ==============================================================
# Đọc bảng tra cứu BTS (T_MASTER_CORD.csv)
bts = pd.read_csv(BTS_PATH, dtype=str)
bts_latest = (
    bts[["AIRPORT_ID", "AIRPORT"]]
    .dropna(subset=["AIRPORT_ID", "AIRPORT"])
    .drop_duplicates(subset="AIRPORT_ID", keep="last")
)
ID_TO_IATA = dict(zip(bts_latest["AIRPORT_ID"], bts_latest["AIRPORT"]))
print(
    f"[3] Đã đọc bảng tra cứu BTS: {len(ID_TO_IATA):,} mã AIRPORT_ID -> IATA."
)


def resolve_airport_code(code: str) -> str:
    """Tra cứu chuyển đổi mã DOT ID dạng số sang mã IATA chuẩn."""
    code = str(code).strip().upper()
    if code in VALID_AIRPORT_CODES:
        return code
    return ID_TO_IATA.get(code, code)


origin_before = df["ORIGIN_AIRPORT"].astype(str).str.strip().str.upper()
dest_before = df["DESTINATION_AIRPORT"].astype(str).str.strip().str.upper()

df["ORIGIN_AIRPORT"] = origin_before.apply(resolve_airport_code)
df["DESTINATION_AIRPORT"] = dest_before.apply(resolve_airport_code)

n_resolved_origin = (origin_before != df["ORIGIN_AIRPORT"]).sum()
n_resolved_dest = (dest_before != df["DESTINATION_AIRPORT"]).sum()
print(
    f"[3] Đã ánh xạ lại (số -> IATA) cho {n_resolved_origin:,} dòng ở ORIGIN, "
    f"{n_resolved_dest:,} dòng ở DESTINATION nhờ bảng BTS."
)

# Flag các mã lạ thực sự không tồn tại ở cả 2 bảng tra cứu
df["ORIGIN_AIRPORT_VALID"] = (
    df["ORIGIN_AIRPORT"].isin(VALID_AIRPORT_CODES).astype(int)
)
df["DEST_AIRPORT_VALID"] = (
    df["DESTINATION_AIRPORT"].isin(VALID_AIRPORT_CODES).astype(int)
)

df["ORIGIN_AIRPORT_CLEAN"] = np.where(
    df["ORIGIN_AIRPORT_VALID"], df["ORIGIN_AIRPORT"], "UNKNOWN"
)
df["DEST_AIRPORT_CLEAN"] = np.where(
    df["DEST_AIRPORT_VALID"], df["DESTINATION_AIRPORT"], "UNKNOWN"
)

n_invalid_origin = (df["ORIGIN_AIRPORT_VALID"] == 0).sum()
n_invalid_dest = (df["DEST_AIRPORT_VALID"] == 0).sum()
print(
    f"[3b] Các mã lạ KHÔNG thể tra cứu: {n_invalid_origin} ở ORIGIN, {n_invalid_dest} ở DESTINATION -> Đánh dấu UNKNOWN."
)

[3] Đã đọc bảng tra cứu BTS: 6,903 mã AIRPORT_ID -> IATA.
[3] Đã ánh xạ lại (số -> IATA) cho 486,165 dòng ở ORIGIN, 486,165 dòng ở DESTINATION nhờ bảng BTS.
[3b] Các mã lạ KHÔNG thể tra cứu: 4078 ở ORIGIN, 4071 ở DESTINATION -> Đánh dấu UNKNOWN.


In [5]:
# ==============================================================
# CELL 5: CHUẨN HÓA DỮ LIỆU LOGIC VÀ XỬ LÝ MISSING VALUES
# ==============================================================
# 1. Chuyển CANCELLED/DIVERTED về kiểu nhị phân (0/1)
df["CANCELLED"] = df["CANCELLED"].astype(int)
df["DIVERTED"] = df["DIVERTED"].astype(int)

# 2. Xử lý thiếu ở các cột lý do trễ cho các chuyến bay HOÀN THÀNH
delay_reason_cols = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
]
mask_completed = df["CANCELLED"] == 0
for c in delay_reason_cols:
    df.loc[mask_completed, c] = df.loc[mask_completed, c].fillna(0)

# 3. Điền giá trị mặc định cho phân loại bị hủy/số hiệu đuôi bay
df["CANCELLATION_REASON"] = df["CANCELLATION_REASON"].fillna("NONE")
df["TAIL_NUMBER"] = df["TAIL_NUMBER"].fillna("UNKNOWN")

# 4. Vá lỗi SCHEDULED_TIME bị thiếu (nếu có)
def hhmm_to_minutes(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x == 2400:
        x = 0
    hh, mm = divmod(x, 100)
    return hh * 60 + mm


n_missing_sched_time = df["SCHEDULED_TIME"].isna().sum()
if n_missing_sched_time > 0:
    mask_missing = df["SCHEDULED_TIME"].isna()
    dep_min = df.loc[mask_missing, "SCHEDULED_DEPARTURE"].apply(hhmm_to_minutes)
    arr_min = df.loc[mask_missing, "SCHEDULED_ARRIVAL"].apply(hhmm_to_minutes)
    df.loc[mask_missing, "SCHEDULED_TIME"] = (arr_min - dep_min) % 1440
print(f"[4] Đã vá {n_missing_sched_time} dòng thiếu SCHEDULED_TIME.")

[4] Đã vá 6 dòng thiếu SCHEDULED_TIME.


In [6]:
# ==============================================================
# CELL 6: TẠO BIẾN TARGET & LƯU FILE ĐÃ LÀM SẠCH TOÀN BỘ
# ==============================================================
# Tạo nhãn mục tiêu IS_DELAYED (Trễ từ 15 phút trở lên)
df["IS_DELAYED"] = np.where(
    df["ARRIVAL_DELAY"].isna(), np.nan, (df["ARRIVAL_DELAY"] >= 15).astype(float)
)

# Cảnh báo Data Leakage
LEAKAGE_COLS = [
    "DEPARTURE_TIME",
    "DEPARTURE_DELAY",
    "TAXI_OUT",
    "WHEELS_OFF",
    "ELAPSED_TIME",
    "AIR_TIME",
    "WHEELS_ON",
    "TAXI_IN",
    "ARRIVAL_TIME",
]
print(
    f"[!] CẢNH BÁO DATA LEAKAGE: Không sử dụng các cột {LEAKAGE_COLS} khi huấn luyện mô hình dự đoán trễ chuyến trước giờ bay!"
)

# Lưu dữ liệu đã làm sạch tổng thể
df.to_csv(CLEANED_PATH, index=False)
print(
    f"[5] Đã lưu file làm sạch thành công: {CLEANED_FILENAME} (Shape: {df.shape})"
)

[!] CẢNH BÁO DATA LEAKAGE: Không sử dụng các cột ['DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'ELAPSED_TIME', 'AIR_TIME', 'WHEELS_ON', 'TAXI_IN', 'ARRIVAL_TIME'] khi huấn luyện mô hình dự đoán trễ chuyến trước giờ bay!
[5] Đã lưu file làm sạch thành công: flights_cleaned.csv (Shape: (5819079, 37))


In [7]:
# ==============================================================
# CELL 7: LỌC DỮ LIỆU THEO HÃNG BAY (TIẾT KIỆM RAM BẰNG CHUNKING)
# ==============================================================
CHUNK_SIZE = 500_000  # Đọc 500,000 dòng/lần

total_rows_read = 0
total_rows_kept = 0
first_chunk = True

print(f"Bắt đầu lọc dữ liệu cho hãng bay '{AIRLINE_CODE}'...")

for chunk in pd.read_csv(CLEANED_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    total_rows_read += len(chunk)

    # Lọc dòng thuộc hãng hàng không chỉ định
    filtered = chunk[chunk["AIRLINE"] == AIRLINE_CODE]
    total_rows_kept += len(filtered)

    # Ghi nối (append) vào file đầu ra
    filtered.to_csv(
        OUTPUT_AIRLINE_PATH,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )
    first_chunk = False
    print(
        f"  - Đã đọc {total_rows_read:,} dòng, giữ lại {total_rows_kept:,} dòng..."
    )

print("\n" + "=" * 50)
print(
    f"HOÀN TẤT: Đã lọc {total_rows_kept:,} / {total_rows_read:,} dòng cho hãng {AIRLINE_CODE}."
)
print(f"File lưu tại: {OUTPUT_AIRLINE_FILENAME}")

Bắt đầu lọc dữ liệu cho hãng bay 'DL'...
  - Đã đọc 500,000 dòng, giữ lại 68,555 dòng...
  - Đã đọc 1,000,000 dòng, giữ lại 140,424 dòng...
  - Đã đọc 1,500,000 dòng, giữ lại 213,164 dòng...
  - Đã đọc 2,000,000 dòng, giữ lại 288,467 dòng...
  - Đã đọc 2,500,000 dòng, giữ lại 364,073 dòng...
  - Đã đọc 3,000,000 dòng, giữ lại 440,469 dòng...
  - Đã đọc 3,500,000 dòng, giữ lại 518,359 dòng...
  - Đã đọc 4,000,000 dòng, giữ lại 597,683 dòng...
  - Đã đọc 4,500,000 dòng, giữ lại 675,386 dòng...
  - Đã đọc 5,000,000 dòng, giữ lại 752,998 dòng...
  - Đã đọc 5,500,000 dòng, giữ lại 829,850 dòng...
  - Đã đọc 5,819,079 dòng, giữ lại 875,881 dòng...

HOÀN TẤT: Đã lọc 875,881 / 5,819,079 dòng cho hãng DL.
File lưu tại: flights_DL.csv
